Ch 1-3: Array, Strings, Lists, Stacks, Queues, Trees, Graphs, etc-- [Open in Colab]([Open in Colab](https://colab.research.google.com/github/henry4j/-/blob/main/episode_1-7.ipynb)

Python Basics: [data structures](https://docs.python.org/3/tutorial/datastructures.html), [deque](https://docs.python.org/3/library/collections.html#collections.deque), [heapq](https://docs.python.org/3/library/heapq.html), [string methods](https://docs.python.org/3.4/library/stdtypes.html#string-methods), [itertools](https://docs.python.org/3/library/itertools.html#itertools-recipes), [functools](https://docs.python.org/3/library/functools.html), [match-case](https://www.inspiredpython.com/course/pattern-matching/mastering-structural-pattern-matching), [new python features](https://www.nicholashairs.com/posts/major-changes-between-python-versions/), [python tricks](https://sahandsaba.com/14-more-python-features-and-tricks-you-may-not-know.html)

---

20 == len(list(permutations(range(5), 2))) and len(combinations(range(5), 2)) == 10

* a k-combination of a set S is a subset of k distinct elements of S, and the # of k-combinations is equals to the binomial coefficient, n! / (k! * (n-k)!).
* a k-permutation of a set S is an ordered sequence of k distinct elements of S, and the # of k-permutation of n objects is denoted variously nPk, Pn,k, and P(n,k), and its value is given by n! / (n-k)!.

In [ ]:
# @title import before you begin
from functools import lru_cache
from itertools import accumulate, islice, chain, count, tee
from collections import namedtuple, Counter, defaultdict, deque
from typing import Optional, Union
from math import *
from random import *
import bisect

def renumerate(it, stop=None):
  return (L := list(it)).reverse() or zip(count(stop or len(L), -1), L)

def recap(kv, k, v):
  return (kv[0] + k, kv[1] + v)

def dedupe(iterable):
  seen = set()
  for e in iterable:
    if e not in seen:
      seen.add(e)
      yield e

In [ ]:
# @title ##### 1.1 Write a program to determine if a string has all unique characters. What if you cannot use additional data structures?
def uniq(s) -> bool:  # time: O(n), space(n)
  return all(v == 1 for v in histogram(s).values())

def histogram(iterable) -> dict:
  h = {}
  for e in iterable:
    h[e] = h.get(e, 0) + 1
  return h

def uniqq(s) -> bool:  # time: o(n^2), space: o(1)
  for i_lhs, lhs in enumerate(s):
    for i_rhs in range(i_lhs + 1, len(s)):
      if lhs == s[i_rhs]:
        return False
  return True


assert all([uniq(""), uniq("a"), uniq("ab"), not uniq("aa"), not uniq("aba")])
assert all([uniqq(""), uniqq("a"), uniqq("ab")])
assert not any([uniqq("aa"), uniqq("aba")])

In [ ]:
# @title ###### 1.2 Write a program to test if two strings are permutations of each other.
def anagram(s1, s2):
    if len(s1) != len(s2):
        return False
    h = histogram(s1)
    for e in s2:
        if h.get(e, 0) > 0:
            h[e] -= 1
        else:
            return False
    return True

def anagram2(s1, s2):
    signature = lambda s: ''.join(sorted(s))
    return len(s1) == len(s2) and signature(s1) == signature(s2)

def anagram3(s1, s2):
    return histogram(s1) == histogram(s2)

assert anagram('', '') and anagram('a', 'a') and anagram('ab', 'ba')
assert anagram('aab', 'aba') and anagram('aabb', 'abab')
assert not anagram('a', '') and not anagram('', 'a')
assert not anagram('a', 'b') and not anagram('aa', 'ab')
assert anagram2('aab', 'aba') and anagram2('aabb', 'abab')
assert anagram3('aab', 'aba') and anagram3('aabb', 'abab')

In [ ]:
# @title ##### 1.3 Write a program to replace all spaces in a string with %20.
def resize(L, new_size, fill_value=None):
  del L[new_size:]
  L.extend([fill_value] * (new_size - len(L)))
  return L

def escape_spaces(s):
  L, m = list(s), len(s)  # m: input length.
  resize(L, n := len(L) + 2 * L.count(" "), " ")  # n: output length.
  w = n
  for r in range(m - 1, -1, -1):  # i in [m-1, 0].
    buff = "%20" if L[r] == " " else [L[r]]
    w -= len(buff)
    L[w : w + len(buff)] = buff
  return "".join(L)

assert "a%20b%20c%20" == escape_spaces("a b c ")

In [ ]:
# @title ##### 1.4 Write a program to test if a string is a permutation of a palindrome.
def histogram(iterable) -> dict:
  h = {}
  for e in iterable:
    h[e] = h.get(e, 0) + 1
  return h

palindormic = lambda s: sum(v % 2 for v in histogram(s.lower()).values()) < 2
assert all([palindormic(""), palindormic("a"), palindormic("aa"), palindormic("aba")])
assert not any([palindormic("ab"), palindormic("abbc")])

In [ ]:
# @title ##### 1.5 Write a program to test if two strings are one or zero edits away from each other (insert, delete, or replace).
from functools import lru_cache
from enum import Enum
Edit = Enum("Edit", {"I": "Insert", "D": "Delete", "R": "Replace", "M": "Match"})

def recap(kv, k, v) -> tuple[int, tuple]:  # k: cost, v: steps.
  return (kv[0] + k, kv[1] + v)

def edit(a, b):
  @lru_cache(maxsize=None)
  def prog(m, n):
    if m == 0 or n == 0:
      return (m + n, ((Edit.D,) * m if m > 0 else (Edit.I,) * n))
    c = int(a[m - 1] != b[n - 1])  # edit cost: 1 or 0.
    cases = (
      recap(prog(m - 1, n - 1), c, ([Edit.M, Edit.R][c],)),
      recap(prog(m - 1, n - 0), 1, (Edit.D,)),
      recap(prog(m - 0, n - 1), 1, (Edit.I,)),
    )
    return min(cases, key=lambda cost_steps: cost_steps[0])
  return prog(len(a), len(b))

expected = (3, (Edit.R, Edit.M, Edit.M, Edit.M, Edit.R, Edit.M, Edit.I))
assert expected == edit("kitten", "sitting")
assert (3, (Edit.D,) * 3) == edit("cat", "")
assert (3, (Edit.I,) * 3) == edit("", "sit")

In [ ]:
# @title ##### 1.6 Write a method to compress a string using counts of repeated chars, e.g., aabcccccaaa becomes a2b1c5a3.
def compressed(s):
  s2, m, start = [], len(s), 0
  for stop in range(1, m + 1):
    if stop == m or s[start] != s[stop]:
      s2.extend([s[start], str(stop - start)])
      start = stop
  return "".join(s2) if len(s2) < m else s

assert "a2b1c5a3" == compressed("aabcccccaaa")
assert "abcc" == compressed("abcc")
assert "abc" == compressed("abc")
assert "" == compressed("")

In [ ]:
# @title ##### 1.7 Given an image represented by an NxN matrix, write a program to rotate the image by 90 degrees; in-place, in O(1) space.
def rotate(g):
  n = len(g)
  for layer in range(n // 2):
    head, tail = layer, n - 1 - layer
    for i in range(head, tail):
      top = g[layer][i]
      g[layer][i] = g[n - 1 - i][head]  # left to top
      g[n - 1 - i][head] = g[tail][n - 1 - i]  # bottom to left
      g[tail][n - 1 - i] = g[i][tail]  # right to bottom
      g[i][tail] = top  # top to right
  return g

g = [
  [1, 2, 3, 4, 5],
  [6, 7, 8, 9, 10],
  [11, 12, 13, 14, 15],
  [16, 17, 18, 19, 20],
  [21, 22, 23, 24, 25],
]
expected = [
  [21, 16, 11, 6, 1],
  [22, 17, 12, 7, 2],
  [23, 18, 13, 8, 3],
  [24, 19, 14, 9, 4],
  [25, 20, 15, 10, 5],
]
assert expected == rotate(g)

In [ ]:
# @title ##### 1.8 Given an NxN matrix, write a program to set the entire row and column to 0 if an element has a value of 0.
def zero_out(g):
  m, n = len(g), len(g[0])
  rows, columns = set(), set()
  for r in range(m):
    for c in range(n):
      if g[r][c] == 0:
        rows.add(r)
        columns.add(c)
  for r in range(m):
    for c in range(n):
      if r in rows or c in columns:
        g[r][c] = 0
  return g

g = [[1, 2, 3, 4], [5, 6, 7, 8], [9, 0, 1, 2], [3, 4, 5, 6]]
assert [[1, 0, 3, 4], [5, 0, 7, 8], [0, 0, 0, 0], [3, 0, 5, 6]] == zero_out(g)

In [ ]:
# @title ##### 1.9 Given two strings, write a program to test if a string is a rotation of the other using isSubstring method.
is_rotated = lambda s, t: len(s) == len(t) and (s + s).find(t) > -1
assert all([is_rotated("x", "x"), is_rotated("xy", "yx"), is_rotated("xyz", "yzx")])
assert not is_rotated("xyz", "xyx")

In [ ]:
# @title ##### 2.1 Write a program to remove duplicates from an unordered linked list. What if you cannot use additional data structures?
from collections import namedtuple
from dataclasses import dataclass

@dataclass
class SNode:
  value: int = None
  next: "SNode" = None

  def __iter__(self):
    L = self
    while L:
      yield L
      L = L.next

  @classmethod
  def from_values(cls, *values):
    L = None
    for value in reversed(values):
      L = cls(value, L)
    return L

def dedup_o_n_time(L):
  curr, seen = L, {}
  while curr:
    if curr.value in seen:
      pred.next = curr.next
    else:
      seen[curr.value], pred = True, curr
    curr = curr.next
  return L

def dedup_o_1_space(L):
  lhs = L
  while lhs:
    pred = lhs  # predecessor of RHS.
    while pred.next:
      if lhs.value == pred.next.value:
        pred.next = pred.next.next
      else:
        pred = pred.next
    lhs = lhs.next
  return L

L3 = SNode.from_values(1, 2, 3)
assert L3 == dedup_o_n_time(SNode.from_values(1, 1, 2, 3, 3))
assert L3 == dedup_o_n_time(SNode.from_values(1, 2, 1, 2, 3))
assert L3 == dedup_o_1_space(SNode.from_values(1, 1, 2, 3, 3))
assert L3 == dedup_o_1_space(SNode.from_values(1, 2, 1, 2, 3))

In [ ]:
# @title ##### 2.2 Implement an algorithm to find the k-th to last element of a singly linked list.
def last(L : SNode, k=0):
  p, pk = L, None  # pk points to the k-th last node when p reaches the end.
  for _ in range(k):
    if p is None:
      break
    p = p.next
  if p:
    pk = L
    while p.next:
      p, pk = p.next, pk.next
  return pk


L4 = SNode.from_values(*range(4))  # 0, 1, 2, 3
assert [3, 2, 0, None] == [(_ := last(L4, k)) and _.value for k in (0, 1, 3, 4)]

In [ ]:
# @title ##### 2.3 Given access only to a node, implement an algorithm to delete that node in the middle of a singly linked list.

In [ ]:
# @title ##### 2.4 Write a program to partition a linked list around a value of x, such that all nodes less than x come before all nodes greater than or equal to x.
def partition(L, x):
  def push(pool, curr):
    curr.next = pool
    return curr
  def last(curr):
    while curr.next:
      curr = curr.next
    return curr
  lt_x, eq_x, gt_x, = [None], [None], [None]  # 1-element containers.
  curr = L
  while curr:
    next_ = curr.next
    pool = lt_x if curr.value < x else gt_x if curr.value > x else eq_x
    pool[0] = push(pool[0], curr)
    curr = next_
  last(lt_x[0]).next, last(eq_x[0]).next = eq_x[0], gt_x[0]
  return lt_x[0]

L9 = SNode.from_values(9, 3, 8, 2, 5, 6, 1, 7, 4, 5)
assert [4, 1, 2, 3, 5, 5, 7, 6, 8, 9] == [e.value for e in partition(L9, 5)]

2.5. Given two single linked lists where each node has a single digit, write a program that sums them up.

In [ ]:
def sum_integer_strings(L, R):  # Sum two integers in strings.
  L, R = (list(reversed(e)) for e in (L, R))  # becomes a: addend and augend
  outcome = []
  tens = 0
  zero = ord("0")
  for i in range(1 + max(len(L), len(R))):
    ones = tens
    ones += sum(ord(a[i]) - zero for a in (L, R) if i < len(a))
    tens, ones = ones // 10, ones % 10
    outcome.append(chr(zero + ones))
  return "".join(reversed(outcome))

assert "1300" == sum_integer_strings("313", "987")

2.6 Write a program to test if a linked list is palindromic.

2.7 Given two linked lists that interacts by reference (not value), write a program to return the intersecting node.

2.8 Given a linked list, implement an algorithm which returns the node at the beginning of the loop. e.g., INPUT: a -> b -> c -> d -> e -> c, and OUTPUT: c.



3.1 How would you design and implement three stacks on a single array.

In [ ]:
# @title ###### 3.2 Design a stack that has **min** method that returns the minimum in addition to push and pop methods. Push, pop, and min should all operate in O(1) time.
class MinStack:
  def __init__(self):
    self.minimum, self.stack = None, []
  def push(self, e):
    if self.minimum is None or e <= self.minimum:
      self.stack.append(self.minimum)  # saves the previous minimum.
      self.minimum = e
    self.stack.append(e)
    return self
  def pop(self):
    if (e := self.stack.pop()) == self.minimum:
      self.minimum = self.stack.pop()
    return e

S = MinStack().push(2).push(3).push(2).push(1)
assert 1 == S.minimum and 1 == S.pop()
assert 2 == S.minimum and 2 == S.pop()
assert 2 == S.minimum and 3 == S.pop()
assert 2 == S.minimum and 2 == S.pop()
assert S.minimum is None

3.3 Imagine a literal stack of plates. If the stack gets too high, it might topple. Therefore, in real life, we would likely start a new stack when the previous stack exceeds some threshold. Implement a data structure SetOfStacks that mimics this. SetOfStacks should be composed of several stacks and should create a new stack once the previous one exceeds capacity. SetOfStacks.push() and SetOfStacks.pop() should behave identically to a single stack (that is, pop() should return the same values as it would if there were just a single stack). Implement a function popAt(int index) which performs a pop operation on a specific sub-stack.

In [ ]:
# @title ##### 3.5 Design a queue using two stacks.
class Q:
  def __init__(self, inbox=[], outbox=[]):
    self.inbox, self.outbox = inbox, outbox
  def offer(self, E):  # E: element.
    self.inbox.append(E)
    return self
  def poll(self):
    if not self.outbox:
      while self.inbox:
        self.outbox.append(self.inbox.pop())
    return self.outbox.pop() if self.outbox else None
  def __repr__(self):
    return f"Q({self.inbox!r}, {self.outbox!r})"

(q := Q()).offer(1).offer(2)
assert 1 == q.poll() and q.offer(3) is q
assert (2, 3, None) == tuple(q.poll() for _ in range(3))

In [ ]:
# @title ##### 3.6 Write a program to sort a stack so the largest elements are at the top. You may use additional stacks to hold items, but you may not copy the elements into any other data structure such as an array. The stack supports the following operations: push, pop, peek, and isEmpty.

def sort(S):  # a = [1, 2, 9, 8, 3, 4]
  L = []
  while S:
    e = S.pop()
    while L and L[-1] > e:
      S.append(L.pop())
    L.append(e)
  S.extend(L)
  return S

assert [1, 2, 3, 4, 5] == sort([1, 2, 5, 3, 4])

In [ ]:
def sort(S):
  L = []
  while S:
    e = S.pop()
    while L and L[-1] > e:
      S.append(L.pop())
    L.append(e)
  return L

3.7 An animal shelter holds only dogs and cats, and operations on a strictly "first in, first out" basis. People must adopt either the oldest (based on the arrival time) of all animals at the shelter, or they can select whether they would prefer a dog or a cat (and will receive the oldest animal of that type). They cannot select which speicific animal they would like. Create the data structures to maintain this system and implement operations such as enqueue, dequeueAny, dequeueDog, and dequeueCat. You may use the LinkedList data structure.

In [ ]:
# @title
def is_prime(n):
  if n in {2, 3}:
    return True
  if n == 1 or n % 2 == 0:
    return False
  for d in range(3, 1 + int(sqrt(n)), 2):
    if n % d == 0:
      return False
  else:
    return True

def prime(n, certainty=5):  # returns when the prob. exceeds 1-0.5 ** certainty.
  if n <4:
    return max(2, n)  # prime p >=2
  if n % 2 == 0:
    n += 1
  for p in range(n, 2 * n):  # prime p exists where n < p <2n-2 when n >1.
    for _ in range(certainty):
      a = 2 + randrange(p-3)  # 2 <= a <= p-2; 2+0 <= a <=2+p-4.
      if 1 != a**(p-1) % p:  # rabin miller's primality test.
        break
    else:
      return p

assert [2, 3, 5, 7, 11] == [e for e in range(12) if is_prime(e)]
assert [2, 2, 2, 3, 5, 5, 7, 7, 11, 11] == [prime(n) for n in range(0, 10)]

In [ ]:
# @title
def factorize(n, m=2):
  if n == 1:
    return tuple()
  elif n % m == 0:
    return (m,) + factorize(n // m)
  elif n < m**2:
    return (n,)
  else:
    return factorize(n, 3 if m == 2 else m+2)

assert (2, 2, 3) == factorize(12) and (2, 3, 7) == factorize(42)
assert (3, 3, 5) == factorize(45) and (3, 5, 5) == factorize(75)

In [ ]:
# @title
inf = 2**31-1  # or sys.maxvalue

def optimal_tour(g):  # TSP http://www.youtube.com/watch?v=aQB_Y9D5pdw
  def popped(s, i):
    l = list(s)
    l.pop(i)
    return tuple(l)

  # g(v, set) is min([g[v, e] + g(e, set-{e}) for e in set])
  def mapped(v, s, memos={}):
    def computed():
      if s:
        return min(((w, g[v][w] + mapped(w, popped(s, i))[1])
                    for i, w in enumerate(s)
                    if g[v][w] != inf),
                   key=lambda e: e[1])
      else:
        return (None, inf) if g[v][0] is inf else (0, g[v][0])

    k = (v, s)
    if k not in memos:
      memos[k] = computed()
    return memos[k]

  return mapped(0, tuple(range(1, len(g))))[1]

g = [None] * 4
g[0] = [0, 10, 15, 20]
g[1] = [5, 0, 9, 10]
g[2] = [6, 13, 0, 12]
g[3] = [8, 8, 9, 0]

assert 35 == optimal_tour(g)

def floyd_warshal(g):
  d = [r[:] for r in g]
  n = len(g)  # graph in n x n matrix
  for k in range(n):
    for i in range(n):
      for j in range(n):
        d[i][j] = min(d[i][j], d[i][k] + d[k][j])
  return d

floyd_warshal(g)

In [ ]:
# @title
def backtrack(candidate, expand_out, reduce_off):
  if not reduce_off(candidate):
    for e in expand_out(candidate):
      candidate.append(e)
      backtrack(candidate, expand_out, reduce_off)
      candidate.pop()

def color_vertex(g):
  def expand_out(a):
    v = len(a)  # vertex v
    return [c for c in set(a) if peaceful_at(a, v, c)] + [max(a)+1]

  def peaceful_at(a, v, c):
    return all([g[v][w] == 0 or a[w] != c for w in range(v)])

  def reduce_off(a):
    if reduce_off.reduced or len(a) == len(g):
      reduce_off.reduced.append((max(a)+1, a[:]))
      return True

  reduce_off.reduced = []
  backtrack([0], expand_out, reduce_off)
  return min(reduce_off.reduced, key=lambda e: e[0])

# http://www.youtube.com/watch?v=Cl3A_9hokjU
g = [None] * 4
g[0] = [0, 1, 0, 1]
g[1] = [1, 0, 1, 1]
g[2] = [0, 1, 0, 1]
g[3] = [1, 1, 1, 0]
assert (3, [0, 1, 0, 2]) == color_vertex(g)

g = [None] * 5
g[0] = [0, 1, 1, 0, 1]
g[1] = [1, 0, 1, 0, 1]
g[2] = [1, 1, 0, 1, 0]
g[3] = [0, 0, 1, 0, 1]
g[4] = [1, 1, 0, 1, 0]
assert (3, [0, 1, 2, 0, 2]) == color_vertex(g)

In [ ]:
# @title
from enum import IntFlag

EType = IntFlag("EType", "ENTER CROSS LEAVE")

class BTree:
  def __init__(self, value=None, left=None, right=None, parent=None):
    self.value, self.left, self.right, self.parent = value, left, right, parent

  def __repr__(self):
    m = len(values := list(vars(self).values()))
    n = next((i for i, e in enumerate(reversed(values)) if e is not None), m)
    return f"BTree({repr(values[:m-n])[1:-1]})"

  def set_parent(self):  # set the parent fields on both the children.
    for e in (self.left, self.right):
      if e:
        e.parent = self
        e.set_parent()

  @classmethod
  def from_values(cls, values, start=0, stop=None):
    if stop is None:
      stop = len(values)
    if stop - start >0:
      mid = (start+stop-1) // 2
      L = BTree.from_values(values, start, mid)
      R = BTree.from_values(values, mid+1, stop)
      return cls(values[mid], L, R, None)

def dfs(tree):
  if tree:
    yield (EType.ENTER, tree)
    yield from chain(dfs(tree.left), dfs(tree.right))
    yield (EType.LEAVE, tree)

def bfs(tree):
  q = deque([tree])
  while q:
    yield (tree := q.popleft())
    for e in (tree.left, tree.right):
      e and q.append(e)

# tree:
#   a
# .   b
#   c
#  d e
B5 = BTree("a", None, BTree("b", BTree("c", BTree("d"), BTree("e"))))
assert "abcde" == "".join(
    tree.value for e_type, tree in dfs(B5) if e_type == EType.ENTER)  # fmt: skip
assert "decba" == "".join(
    tree.value for e_type, tree in dfs(B5) if e_type == EType.LEAVE)  # fmt: skip
assert "abcde" == "".join(tree.value for tree in bfs(B5))

In [ ]:
# @title
from itertools import islice

def iterate(tree):  # returns an iterator.
  stack = []
  while tree or stack:
    if tree:
      stack.append(tree)
      tree = tree.left
    else:
      yield (tree := stack.pop())
      tree = tree.right

def riterate(tree):  # returns a reverse iterator.
  stack = []
  while tree or stack:
    if tree:
      stack.append(tree)
      tree = tree.right
    else:
      yield (tree := stack.pop())
      tree = tree.left

T10 = BTree(
  4,
  BTree(3, BTree(2, BTree(1, BTree(0)))),
  BTree(8, BTree(5, None, BTree(7, BTree(6))), BTree(9)))  # fmt: skip

assert list(range(10)) == list(tree.value for tree in iterate(T10))
assert [9, 8] == list(e.value for e in islice(riterate(T10), 2))
assert [8, 7] == list(e.value for e in islice(riterate(T10), 1, 3))
assert [8, 6, 4] == list(e.value for e in islice(riterate(T10), 1, 6, 2))

Q: Is a graph a tree? How to find a cycle if one exists? deadlocked?

* undirected: it is a tree if the undirected graph is connected and has n - 1 edges for n vertices.
* directed: a directed acyclic graph has no back edges; a cyclic graph has back edges.

In [ ]:
# @title
# fmt: off
"""graph:
A0 →  C2 → E4
↓  ↗  ↓   ↗
B1  . D3"""
a, b, c, d, e = range(5)
g = [
  {b, c},  # from a
  {c},  # from b
  {d, e},  # from c
  {e},  # from d
  {}  # from e
]
# fmt: on

def has_cycle(g, directed=False):
  def dfs(x, entered, exited, tree_edges, back_edges):
    if x not in entered:
      entered.add(x)
      for y in g[x]:
        if y not in entered:
          tree_edges[y] = x
        elif (
          not directed
          and tree_edges.get(x, None) != y
          or directed
          and y not in exited
        ):
          back_edges.setdefault(y, set()).add(x)
        dfs(y, entered, exited, tree_edges, back_edges)
      exited.add(x)
    return (tree_edges, back_edges)
  for x in range(len(g)):
    if dfs(x, entered=set(), exited=set(), tree_edges={}, back_edges={})[1]:
      return True
  else:
    return False

assert not has_cycle(g, True)
g[a] = {b}
g[c] = {a, d, e}
assert has_cycle(g, True)
# undirected graph: A0 - B1 - C2
a, b, c = range(3)
g2 = [{b}, {a, c}, {b}]
assert not has_cycle(g2, False)
# undirected graph: A0 - B1 - C2 - A0
g2[a].add(c)
g2[c].add(a)
assert has_cycle(g2, False)

In [ ]:
# @title ##### 4.1 Given a directed graph, design an algorithm to find whether there is a route between two nodes.
from collections import namedtuple

class Edge(namedtuple("Edge", ["to", "weight"], defaults=[0])):
  def __repr__(self):
    return f"Edge({self.to!r}{(w := self.weight) and f', {w!r}' or ''})"

def DFS(graph, vertex, entered=None):
  entered = set() if entered is None else entered
  if vertex not in entered and not entered.add(vertex):
    # yield EType.ENTER, (edge := Edge(v)), None
    for e in graph[vertex] or []:
      yield EType.CROSS, e, vertex
      yield from DFS(graph, e.to, entered)
    yield EType.LEAVE, Edge(vertex), None

# graph:
# B1 ← C2 → A0
# ↓  ↗
# D3 ← E4
graph = [[]] * 5
graph[0] = []  # out-degree of 0
graph[1] = [Edge(3)]  # B1 → D3
graph[2] = [Edge(0), Edge(1)]  # C2 → A0, C2 → B1
graph[3] = [Edge(2)]  # D3 → C2
graph[4] = [Edge(3)]  # E4 → D3

def can_reach(source, sink, graph) -> bool:
  return any(edge.to == sink for type_, edge, from_ in DFS(graph, source))

assert all(can_reach(4, sink, graph) for sink in range(3))
assert not any(can_reach(source, 4, graph) for source in (0, 3))

In [ ]:
# @title ##### 4.2 Given a sorted (increasing order) array, design an algorithm to create a binary search tree with the minimal height.
# tree:
# .  4
#   / \
#  2   6
# 1 3 5 7
T7 = BTree.from_values((1, 2, 3, 4, 5, 6, 7))
T7_repr = "BTree(4, BTree(2, BTree(1), BTree(3)), BTree(6, BTree(5), BTree(7)))"
assert T7_repr == repr(T7)

In [ ]:
# @title ##### 4.3 Given a binary tree, design an algorithm to creates a linked list of all the nodes at each depth, e.g., if you have a tree with depth D, you will have D linked lists.
def to_d_linked_lists(tree):  # tree of depth d.
  LL = []  # output: a list of lists.
  q = [tree]
  while q:
    p = []
    for i, e in enumerate(q):
      for c in (e.left, e.right):
        if c:
          p.append(c)
      e.left = None if i == 0 else q[i - 1]
      e.right = q[i + 1] if i < len(q) else None
    LL.append(q[0])
    q = p
  return LL

In [ ]:
# @title ##### 4.4 Implement a function to check if a binary tree is balanced. For the purposes of this question, a balanced tree is defined to be a tree such that the heights of two subtrees of any node never differ by more than one.
# tree: a
# . .  /   \
# .   b  .  f
#   c  e   g
#  d
c = BTree("c", BTree("d"))
e = BTree("e")
b = BTree("b", c, e)
a = BTree("a", b, BTree("f"))

def is_balanced(tree):  # returns (balanced, height)
  if tree is None:
    return (True, -1)
  L, R = is_balanced(tree.left), is_balanced(tree.right)
  b = L[0] and R[0] and abs(L[1] - R[1]) < 2
  h = 1 + max(L[1], R[1])
  return (b, h)

assert not is_balanced(a)[0]
a.right.left = BTree("g")
assert is_balanced(a)[0]

In [ ]:
# @title ##### 4.5 Implement a function to check if a binary tree is a binary search tree.
def order_(tree):
  if tree:
    yield from chain(order_(tree.left), (tree,), order_(tree.right))

def is_ordered(tree):
  pred = next(iterator := iterate(tree))
  for e in iterator:
    if e.value > pred.value:
      return e
    else:
      pred = e

def ordered(tree):  # returns (ordered, min, max)
  if tree is None:
    return (True, None, None)
  L, R = ordered(tree.left), ordered(tree.right)
  O = (
    L[0]
    and R[0]
    and (L[2] is None or L[2] <= tree.value)
    and (R[1] is None or R[1] >= tree.value)
  )
  min_ = tree.value if L[1] is None else L[1]
  max_ = tree.value if R[2] is None else R[2]
  return (O, min_, max_)

B7 = BTree.from_values([1, 2, 3, 4, 5, 6, 7])
assert is_ordered(B7)
assert ordered(B7)[0]
assert not ordered(BTree.from_values((1, 2, 3, 4, 0, 6, 7)))[0]

In [ ]:
# @title ##### 4.6 Design an algorithm to find the next node (i.e., successor in-order) of a node in a binary search tree. You may assume that each node has a link to its parents.
# tree: .  z
#  .  .  u
#  .  .  . v
#  .  .  .   y
#  .  .  . x
#  .  .  w
w = BTree("w")
x = BTree("x", w)
y = BTree("y", x)
v = BTree("v", None, y)
u = BTree("u", None, v)
z = BTree("z", u)
z.set_parent()

def succ(node):
  if node is None:
    raise RuntimeError("'node' must be non-null.")
  if node.right:
    node = node.right
    while node.left:
      node = node.left
    return node
  else:
    while node.parent and node == node.parent.right:
      node = node.parent
    return node.parent

assert v == succ(u) and w == succ(v) and x == succ(w)
assert y == succ(x) and z == succ(y) and succ(z) is None

In [ ]:
# @title
from collections import deque

class BTree:
  def dfs(self, may_enter=None, leave=None):  # for traversals
    if not may_enter or may_enter(self):
      for w in (self.left, self.right):
        w and w.dfs(may_enter, leave)
      leave and leave(self)
  def bfs(self, may_enter=None, leave=None):
    may_enter = may_enter or (lambda *_, **__: True)
    q = deque()
    q.append(self)  # enque, or offer
    while q:
      v = q.popleft()  # deque, or poll
      if may_enter(v):
        for w in (v.left, v.right):
          w and q.append(w)
      leave and leave(v)

In [ ]:
# @title ##### 4.7 Given a set of projects and their dependencies. Find a build order that ensures each project is built only after its dependencies are built.
def topological_sort(graph):
  entered = set()
  for vertex, _ in enumerate(graph):
    if vertex not in entered:
      for e_type, edge, from_ in DFS(graph, vertex, entered):
        if e_type == EType.LEAVE:
          yield edge.to

# graph:   .  D3 ⇾ H7
#  .  .  .  . ↑
#  ┌──────-── B1 ⇾ F5
#  ↓  . .  .  ↑  .  ↑
#  J9 ⇽ E4 ⇽ A0 ⇾ C2 ⇾ I8
#  ↓
#  G6
graph = [[]] * 10
graph[0] = [Edge(1), Edge(2), Edge(4), Edge(6)]  # 1, 2, 4, and 6 depend on 0.
graph[1] = [Edge(3), Edge(5), Edge(9)]  # 3, 5, and 9 depend on 1.
graph[2] = [Edge(5), Edge(8)]  # 5, 8 depend on 2.
graph[3] = [Edge(7)]  # 7 depend on 3.
graph[4] = [Edge(9)]  # 9 depend on 4.
sort = deque(topological_sort(graph))
assert 10 == len(sort) and 0 == sort[-1]

In [ ]:
# @title ##### 4.8 Design an algorithm to find the first common ancestor of the nodes in a binary tree. Avoid storing additional nodes in a data structure. Note: this is not necessarily a binary search tree.
def is_subtree(tree, subtree):
  return (
    tree == subtree
    or tree
    and (is_subtree(tree.left, subtree) or is_subtree(tree.right, subtree))
  )

def lowest_common_ancestor(tree, p, q):
  if is_subtree(tree, p) and is_subtree(tree, q):
    while tree:
      if tree is p or tree is q:
        return tree
      p_on_left = is_subtree(tree.left, p)
      q_on_left = is_subtree(tree.left, q)
      if p_on_left != q_on_left:
        return tree
      tree = tree.left if p_on_left else tree.right

def lowest_common_ancestor2(tree, p, q):  # returns (count, LCA)
  if tree is None:
    return (None, 0)
  count = (p, q).count(tree)
  if count == 2:
    return (tree, 2)
  l = lowest_common_ancestor2(tree.left, p, q)
  if l[1] == 2:
    return l
  r = lowest_common_ancestor2(tree.right, p, q)
  if r[1] == 2:
    return r
  count += l[1] + r[1]
  return (tree if count == 2 else None, count)

# tree:.   a
#  .  .  /   \
#  .   b  .   f
#  .  c  e  g
#   d
d = BTree("d")
c = BTree("c", d)
e = BTree("e")
b = BTree("b", c, e)
a = BTree("a", b, BTree("f", BTree("g")))
assert is_subtree(a, a)
assert is_subtree(a, a.left) and is_subtree(a, a.left.right)
assert is_subtree(a, a.right) and is_subtree(a, a.right.left)
assert b == lowest_common_ancestor(a, d, e)
assert b == lowest_common_ancestor2(a, d, e)[0]

In [ ]:
# @title
def parse(preorder, inorder):
  if preorder:
    v = preorder[0]
    k = inorder.index(v)
    return BNode(v, parse(preorder[1:k+1], inorder[:k]),
                 parse(preorder[k+1:], inorder[k+1:]))

def diameter(tree):  # return (diameter, height).
  if tree is None:
    return (0, 0)
  L = diameter(tree.left)
  R = diameter(tree.right)
  d = max(L[0], R[0], 1 + L[1] + R[1])
  height = 1 + max(L[1], R[1])
  return (d, height)

# tree input:   a
#             b
#          c    f
#           d     g
#            e
tree = parse("abcdefg", "cdebfga")
assert (6, 5) == diameter(tree)

In [ ]:
# @title ##### 4.9 Given a binary search tree that was reconstructed from an array of integers, find all the possible arrays that could have led to this binary search tree.
from functools import lru_cache
from itertools import chain

def weave(L, R):
  @lru_cache(maxsize=None)
  def prog(m, n):
    if m == 0 or n == 0:
      yield L[:m] + R[:n]
    else:
      yield from chain(
        (e + [L[m - 1]] for e in prog(m - 1, n)),
        (e + [R[n - 1]] for e in prog(m, n - 1)),
      )
  yield from prog((m := len(L)), (n := len(R)))

def destruct(tree: BTree):
  if tree is None:
    yield []
  else:
    yield from (
      [tree.value] + e
      for L in destruct(tree.left)
      for R in destruct(tree.right)
      for e in weave(L, R)
    )

T5 = BTree(4, BTree(1, None, BTree(3, BTree(2))), BTree(5))
expected = ([4, 5, 1, 3, 2], [4, 1, 5, 3, 2], [4, 1, 3, 5, 2], [4, 1, 3, 2, 5])
assert expected == tuple(destruct(T5))

In [ ]:
# @title ##### 4.10 You have two very large binary trees: T1 with millions of nodes, and T2 with hundreds of nodes. Design an algorithm to decide if T2 is a subtree of T1. A tree T2 is a subtree of T1 if there exists a node in T1 such that the subtree of n is identical to T2. i.e., if you cut off the tree at node n, the two trees would be identical.
def contains(tree, subtree):
  return (
    subtree is None
    or starts_with(tree, subtree)
    or (tree and (contains(tree.left, subtree) or contains(tree.right, subtree)))  # fmt: skip
  )

def starts_with(tree, subtree):
  return (
    subtree is None
    or tree
    and tree.value == subtree.value
    and starts_with(tree.left, subtree.left)
    and starts_with(tree.right, subtree.right)
  )

assert contains(a, None) and contains(a, a)
assert contains(a, BTree("b", BTree("c")))
assert contains(a, BTree("b", None, BTree("e")))
assert contains(a, BTree("b", BTree("c"), BTree("e")))

In [ ]:
# @title ##### 4.11 Design a binary search tree capable of selecting a random node with equal probability. Operations such as inserting, deleting, finding, and randomly selecting a node should have sublinear time complexity.
class BNode:
  def __init__(self, key=None, left=None, right=None, value=None):
    self.key, self.left, self.right = key, left, right
    self.value = value
    size_sum = sum(c and c.size or 0 for c in (self.left, self.right))
    self._size = size_sum + 1 if size_sum > 0 else None
  @property
  def size(self):
    return self._size or 1
  def __repr__(self):
    m = len(values := list(v for (k, v) in vars(self).items() if k[0] != "_"))
    n = next((i for i, e in enumerate(reversed(values)) if e is not None), m)
    return f"BNode({repr(values[:m-n])[1:-1]})"
  def __setitem__(self, key, value):
    lookup = self.lookup(key)
    e = next(lookup, None)
    if e.key == key:
      e.value = value
      return
    elif key < e.key:
      e.left = BNode(key, value=value)
    else:
      e.right = BNode(key, value=value)
    e._size = e.size + 1
    for e in lookup:
      e._size = e.size + 1
  def __getitem__(self, key):
    return e if (e := next(self.lookup(key))).key == key else None
  def lookup(self, key):
    if key < self.key and self.left:
      yield from self.left.lookup(key)
    elif key > self.key and self.right:
      yield from self.right.lookup(key)
    yield self
  def at(self, index: int):
    L_size = (L := self.left) and L.size or 0
    if index > L_size:
      return self.right.at(index - 1 - L_size) if self.right else None
    elif index < L_size:
      return self.left.at(index) if self.left else None
    else:
      return self

# tree:
# .  5
#   /  \
#  3  . 8
# 2 4  7 9
# . . 6
B6 = BNode(5, BNode(3, BNode(2), BNode(4)), BNode(8, BNode(7)))
B6[6], B6[9] = None, None
expected = [2, 3, 4, 5, 6, 7, 8, 9, None]
assert expected == [(n := B6.at(i)) and n.key for i in range(9)]

In [ ]:
# @title ##### 4.12 Given a binary tree in which each node contains a value. Design an algorithm to print all paths which sum to a given value. Note that a path can start or end anywhere in the tree.
# tree: -1
#     ↘
#       3
#     ↙
#     -1
#    ↙ ↘
#   2  3
def path_of_sum(node, sum, breadcrumbs=None, prefix_sums=None, sum_begins_from=None):
  paths = []
  if node:
    breadcrumbs = breadcrumbs or []
    prefix_sums = prefix_sums or []
    sum_begins_from = sum_begins_from or {sum: [0]}
    breadcrumbs.append(node.value)
    prefix_sums.append(node.value + (prefix_sums[-1] if prefix_sums else 0))
    sum_begins_from.setdefault(prefix_sums[-1] + sum, []).append(len(breadcrumbs))
    for e in sum_begins_from.get(prefix_sums[-1], []):
      yield "-> ".join(map(str, breadcrumbs[e:]))
    yield from chain(
        path_of_sum(node.left, sum, breadcrumbs, prefix_sums, sum_begins_from),
        path_of_sum(node.right, sum, breadcrumbs, prefix_sums, sum_begins_from))
    sum_begins_from[prefix_sums[-1] + sum].pop()
    prefix_sums.pop()
    breadcrumbs.pop()

tree = BTree(-1, None, BTree(3, BTree(-1, BTree(2), BTree(3))))
expected = ["-1-> 3", "3-> -1", "2", "-1-> 3"]
assert expected == list(path_of_sum(tree, 2))

In [ ]:
# @title ##### 5.2 Given a double-precision floating-point number between 0 and 1, output its binary representation. If the exact binary representation requires more than 32 bits, print an error message.
def float_to_str(f: float) -> str:
  if f <= 0 or f >= 1:
    return "ERROR"
  L = list(".")
  while f > 0:
    if len(L) > 31:
      return "ERROR"
    f = round(2 * f, 6)
    if f >= 1:
      L.append("1")
      f -= 1
    else:
      L.append("0")
  return "".join(L)

assert ".011" == float_to_str(0.375)
assert ".101" == float_to_str(0.625)

In [ ]:
# @title ##### 5.3 Given an integer, write a program to find the length of the longest sequence of 1s after flipping exactly one 0 to 1.

In [ ]:
# @title ##### 5.4 Given a positive integer, print the predecessor number (largest, smaller), and the successor number (smallest, larger) that have the same number of 1 bits.
def succ(n: int) -> int:
  # count 0s, and 1s
  m = n.bit_length()
  c0 = next((c0 for c0 in range(m) if n & (1 << c0)), 0)  # c0: no. 0s.
  c1 = next((c1 for c1 in range(c0, m) if n & (1 << c1) == 0), m) - c0  # fmt: skip
  return n + (1 << c0) + (1 << c1 - 1) - 1

def pred(n: int) -> int:
  # count 1s and 0s.
  m = n.bit_length()
  c1 = next((c1 for c1 in range(m) if n & (1 << c1) == 0), m)  # c1: no. 1s.
  c0 = next((c0 for c0 in range(c1, m) if n & (1 << c0)), 0) - c1  # c0: no. 0s.
  return n - (1 << c1) - (1 << c0 - 1) + 1

assert 284 == succ(256 + 16 + 8 + 2)
assert 256 + 16 + 8 + 2 == pred(284)

5.5 What does this code do? (n > 0 and n & (m-1) == 0)

5.6 Write a program to get the Hamming distance between two integers -- the number of positions at which the corresponding bits are different. `(lhs ^ rhs).bit_count()`.


5.7 Write a program to swap odd and even bits in an integer with as few instructions as possible.  
`swap_bits = lambda n: (n & 0xAAAAAAAA) >> 1 | (n & 0x55555555) << 1`

In [ ]:
# @title ##### 5.8 Design an algorithm to draw a horizontal line on a monochrome screen, where the screen is represented as a byte array with each byte storing 8 pixels. The function signature looks like: draw_line(screen: bytes[], width, x1, x2, y: int)